# 02 — Record the FT232H / MPU-6050 FIFO

Connect the FT232H to the MPU-6050 using your working I²C wiring and
PyFtdi driver setup. The standalone CLI is `python record_ftdi.py`.

**Timing:** at DLPF_CFG=3, `SMPLRT_DIV=4` implies a nominal 200 Hz sensor
sampling rate. All FIFO samples are assigned `sample_index / 200` seconds.
Python decoding time is **not** used as sample time. Without a per-sample
hardware timestamp, oscillator error cannot be recovered.

**Overflow:** on a latched FIFO overflow we stop the recording and write
metadata identifying it as invalid for contiguous inertial integration.
`Ctrl+C` in this cell returns partial samples and closes the USB interface.
Remove the example calibration if you have not recorded a real one.

In [ ]:
try:
    from pathlib import Path
    from imu_lab.calibration import Calibration, calibration_callback
    from imu_lab.ftdi_mpu6050 import MPU6050Ftdi
    from imu_lab.recording import save_capture
    FTDI_URL = 'ftdi://ftdi:232h/1'
    RECORD_SECONDS = 20.0
    CALIBRATION_FILE = Path('data/calibration.json')
    OUT_CSV = Path('data/stationary.csv')
    calibration = Calibration.from_json(CALIBRATION_FILE) if CALIBRATION_FILE.exists() else None
    if calibration is None:
        print('No real calibration found: recording uncalibrated SI measurements.')
    print('Output:', OUT_CSV)
except KeyboardInterrupt:
    print("Cell interrupted safely; kernel stays usable.")

In [ ]:
try:
    capture_result = None
    # The `with` block guarantees terminate() even if the kernel is interrupted.
    # The capture method catches Ctrl+C and returns all complete received packets.
    with MPU6050Ftdi(url=FTDI_URL, sample_div=4, dlpf_cfg=3,
                     calibration=calibration_callback if calibration else None,
                     calibration_data=calibration) as sensor:
        capture_result = sensor.capture(duration_s=RECORD_SECONDS)
    if capture_result and capture_result.samples:
        save_capture(capture_result, OUT_CSV, calibrated=calibration is not None)
        print('Saved', len(capture_result.samples), 'complete samples.',
              'Stop:', capture_result.stop_reason)
    else:
        print('No samples to save.')
    if capture_result and capture_result.overflow_detected:
        print('OVERFLOW! Discard this recording for integration; shorten polling/recording.')
except KeyboardInterrupt:
    print("Cell interrupted safely; kernel stays usable.")